In [2]:
import os
import sys
import warnings

# ===== PASTE YOUR HF TOKEN HERE =====
HF_TOKEN = "<>>"
# ====================================

os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

warnings.filterwarnings('ignore')

if HF_TOKEN and HF_TOKEN != "your_token_here":
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✓ HF_TOKEN set successfully")
else:
    print("⚠ HF_TOKEN not set. Add your token above.")

# Load the model and tokenizer here so the rest of the notebook works after Cell 1.
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"✓ Model and tokenizer loaded: {model_name}")

✓ HF_TOKEN set successfully
✓ Model and tokenizer loaded: Qwen/Qwen2.5-Coder-3B-Instruct


In [ ]:
# This cell is skippable


sample = {
    "en": "Open the consumption model containing the measures and attributes you want to include in your perspective, and click the Perspectives tab.",
    "proper_terms": {
        "consumption model": "Verbrauchsmodell"
    },
    "random_terms": {
        "include": "aufnehmen",
        "want": "möchten"
    }
}

prompt = f"""
You are a translation assistant.

Translate the English text to German.

Rules:
1. Output only in this format: <deu> ... </deu>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.

Terminology:
{sample["proper_terms"]}
{sample["random_terms"]}

Input:
<en> {sample["en"]} </en>
"""

messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

In [3]:
# Cell: install dependencies and download data from Google Drive folder
# Run this cell once in Colab to fetch data files into `data/`

# Install required packages
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "gdown"]) 

import os
import gdown

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Google Drive folder link (shared). Downloads into `data/` folder.
FOLDER_URL = "https://drive.google.com/drive/folders/1-O3X6YCv9j0dn0RUtGBe6yEBwnZvgpWD"
print("Downloading files from Google Drive folder. This may ask for confirmation in Colab logs.")
try:
    gdown.download_folder(FOLDER_URL, output=DATA_DIR, quiet=False, use_cookies=False)
except Exception as e:
    print("gdown.download_folder failed:", e)
    print("If running in Colab, consider mounting Drive and copying files manually:")
    print("from google.colab import drive; drive.mount('/content/drive')")

print("Data files in data/:", os.listdir(DATA_DIR))

Retrieving folder contents


Processing file 1CAJ5bpUgxw62rg0dBpuvJB1rTOem2wit ende_dev.jsonl
Processing file 1KuDfHwokf8Z2XE1uGVWrvNwlWqRgss7u enes_dev.jsonl
Processing file 1uKu_nH_p_ByEViYCCnFAeuSkJkENm74M enru_dev.jsonl


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1CAJ5bpUgxw62rg0dBpuvJB1rTOem2wit
To: /content/data/ende_dev.jsonl
100%|██████████| 127k/127k [00:00<00:00, 54.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KuDfHwokf8Z2XE1uGVWrvNwlWqRgss7u
To: /content/data/enes_dev.jsonl
100%|██████████| 127k/127k [00:00<00:00, 66.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1uKu_nH_p_ByEViYCCnFAeuSkJkENm74M
To: /content/data/enru_dev.jsonl
100%|██████████| 155k/155k [00:00<00:00, 71.9MB/s]

Data files in data/: ['enes_dev.jsonl', 'enru_dev.jsonl', 'ende_dev.jsonl']



Download completed


In [7]:
# Cell: translation helper functions
import os
import json
import re
from typing import List, Dict, Optional


def load_jsonl(path: str) -> List[Dict]:
    out = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            out.append(json.loads(line))
    return out


def save_jsonl(path: str, records: List[Dict]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def detect_target_language(samples: List[Dict]) -> tuple:
    """Detect target language from sample data.
    Returns: (lang_code, lang_name, output_tag, ref_field)
    """
    for sample in samples:
        if 'de' in sample:
            return ('de', 'German', 'deu', 'de')
        if 'es' in sample:
            return ('es', 'Spanish', 'es', 'es')
        if 'ru' in sample:
            return ('ru', 'Russian', 'ru', 'ru')
    return ('de', 'German', 'deu', 'de')


def strip_output_tags(text: str) -> str:
    """Remove <deu>/<es>/<ru> tags from model outputs for scoring/matching."""
    if not isinstance(text, str):
        return text
    return re.sub(r'</?(deu|es|ru)>', '', text, flags=re.IGNORECASE).strip()


def translate_sample(
    model,
    tokenizer,
    sample_en: str,
    terminology: Optional[Dict[str, str]] = None,
    target_lang: str = 'German',
    output_tag: str = 'deu',
    max_new_tokens: int = 256
) -> str:
    term_block = ""
    if terminology:
        term_block = "Terminology:\n"
        for k, v in terminology.items():
            term_block += f"{k} -> {v}\n"
        term_block += "\n"

    prompt = f"""
You are a translation assistant.

Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.

{term_block}
Input:
<en> {sample_en} </en>
"""

    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response


# Batch wrapper
def translate_batch(
    model,
    tokenizer,
    samples: List[Dict],
    mode: str = 'no_term',
    target_lang: str = 'German',
    output_tag: str = 'deu'
) -> List[str]:
    preds = []
    for sample in samples:
        if mode == 'no_term':
            terminology = None
        elif mode == 'proper_term':
            terminology = sample.get('proper_terms') or None
        elif mode == 'random_term':
            terminology = sample.get('random_terms') or None
            if terminology and sample.get('proper_terms'):
                for k in sample['proper_terms'].keys():
                    terminology.pop(k, None)
        else:
            terminology = None

        pred = translate_sample(
            model,
            tokenizer,
            sample['en'],
            terminology=terminology,
            target_lang=target_lang,
            output_tag=output_tag
        )
        preds.append(pred)
    return preds

In [ ]:
# Cell: metrics and terminology scoring helpers (Paper §4.2-4.3)
import sacrebleu
from collections import defaultdict, Counter
import re


def compute_bleu_chrf(hyps: List[str], refs: List[str]):
    # BLEU: corpus-level n-gram precision with brevity penalty (sacrebleu.corpus_bleu)
    # chrF2++: character n-gram F-score (sacrebleu.corpus_chrf)
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {"bleu": bleu.score, "chrf": chrf.score}


def _normalize_text(text: str) -> str:
    """Lowercase and remove extra whitespace for matching."""
    return " ".join(text.lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    """Count occurrences of term in text (case-insensitive, whole-word match)."""
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    pattern = r'\b' + re.escape(term_norm) + r'\b'
    return len(re.findall(pattern, text_norm))


def terminology_accuracy_advanced(preds: List[str], samples: List[Dict], mode: str = 'proper_term') -> Dict:
    # For each source term: ratio = min(target_count / source_count, 1.0)
    # Aggregate by averaging ratios across all term types (paper).
    term_ratios = {}
    total_terms = 0

    for pred, sample in zip(preds, samples):
        if mode == 'proper_term':
            terms = sample.get('proper_terms') or {}
        elif mode == 'random_term':
            terms = sample.get('random_terms') or {}
            for k in (sample.get('proper_terms') or {}).keys():
                terms.pop(k, None)
        else:
            terms = {}

        source_text = sample.get('en', '')
        for src, tgt in terms.items():
            total_terms += 1
            src_count = _count_term_occurrences(source_text, src)
            if src_count == 0:
                src_count = 1
            tgt_count = _count_term_occurrences(pred, tgt)
            ratio = min(tgt_count / src_count, 1.0)
            term_ratios[src] = ratio

    avg_accuracy = (sum(term_ratios.values()) / len(term_ratios) * 100) if term_ratios else None
    return {"total_terms": total_terms, "avg_ratio_pct": avg_accuracy, "per_term_ratios": term_ratios}


def terminology_consistency_advanced(preds: List[str], samples: List[Dict], mode: str = 'proper_term') -> Dict:
    # Build pseudo-reference per source term = most frequent predicted target
    # Consistency = fraction of occurrences matching the pseudo-reference (Paper).
    term_to_candidates = defaultdict(list)

    for pred, sample in zip(preds, samples):
        if mode == 'proper_term':
            terms = sample.get('proper_terms') or {}
        elif mode == 'random_term':
            terms = sample.get('random_terms') or {}
            for k in (sample.get('proper_terms') or {}).keys():
                terms.pop(k, None)
        else:
            terms = {}

        for src, tgt in terms.items():
            if tgt.lower() in pred.lower():
                term_to_candidates[src].append(tgt)
            else:
                term_to_candidates[src].append('<MISSING>')

    pseudo_references = {}
    for src, candidates in term_to_candidates.items():
        cnt = Counter(candidates)
        most_common = cnt.most_common(1)[0][0]
        pseudo_references[src] = most_common

    per_term_consistency = {}
    macro_scores = []
    weighted_scores = []

    for src, candidates in term_to_candidates.items():
        pseudo_ref = pseudo_references[src]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates) if candidates else 0.0
        per_term_consistency[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency
        }
        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))

    # Macro: equal weight per term type
    macro_avg = sum(macro_scores) / len(macro_scores) if macro_scores else None
    # Weighted: weight by number of occurrences
    weighted_avg = sum(weighted_scores) / len(weighted_scores) if weighted_scores else None
    return {
        "per_term": per_term_consistency,
        "macro_avg_consistency": macro_avg,
        "weighted_avg_consistency": weighted_avg
    }

In [ ]:
# Cell: evaluation runner — run modes and print summary metrics in notebook output
import glob
import subprocess
import sys

# Optional: set max samples to process per file (None = all)
MAX_SAMPLES = None  # e.g. 100 or None
TOP_TERM_PREVIEW = 5
TOP_SAMPLE_PREVIEW = 2

# Ensure tqdm available
try:
    from tqdm import tqdm
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm", "-q"])
    from tqdm import tqdm

files = sorted([p for p in glob.glob('data/*.jsonl') if p.endswith('_dev.jsonl')])
print('Found data files:', files)

modes = ['no_term', 'proper_term', 'random_term']


def _fmt_metric(value, digits=2):
    if value is None:
        return 'N/A'
    return f'{value:.{digits}f}'


def _preview_term_stats(term_stats, limit=5):
    if not term_stats:
        return []
    items = sorted(term_stats.items(), key=lambda item: (-item[1].get('occ', 0), item[0]))
    return [(term, stats) for term, stats in items[:limit]]


for filepath in files:
    samples = load_jsonl(filepath)
    if MAX_SAMPLES is not None:
        samples = samples[:MAX_SAMPLES]

    lang_code, lang_name, output_tag, ref_field = detect_target_language(samples)
    print(f'\n=== File: {filepath} | target={lang_name} | samples={len(samples)} | MAX_SAMPLES={MAX_SAMPLES} ===')

    for mode in modes:
        print(f'\n--- Mode: {mode} ---')
        preds = []
        desc = f'{os.path.basename(filepath)} - {mode}'

        for s in tqdm(samples, desc=desc):
            if mode == 'no_term':
                terminology = None
            elif mode == 'proper_term':
                terminology = s.get('proper_terms') or None
            elif mode == 'random_term':
                terminology = (s.get('random_terms') or {}).copy()
                for k in (s.get('proper_terms') or {}).keys():
                    terminology.pop(k, None)
                if not terminology:
                    terminology = None
            else:
                terminology = None

            pred = translate_sample(
                model,
                tokenizer,
                s.get('en', ''),
                terminology=terminology,
                target_lang=lang_name,
                output_tag=output_tag
            )
            preds.append(pred)

        refs = [s.get(ref_field) for s in samples]
        clean_preds = [strip_output_tags(p) for p in preds]
        metrics = compute_bleu_chrf(clean_preds, refs)

        term_mode = 'proper_term' if mode == 'proper_term' else 'random_term' if mode == 'random_term' else 'no_term'
        term_acc = terminology_accuracy_advanced(clean_preds, samples, mode=term_mode)
        term_cons = terminology_consistency_advanced(clean_preds, samples, mode=term_mode)

        print(f"BLEU: {_fmt_metric(metrics['bleu'])}")
        print(f"chrF2++: {_fmt_metric(metrics['chrf'])}")
        print(f"Terminology accuracy (ratio %): {_fmt_metric(term_acc.get('avg_ratio_pct'))}")
        print(f"Terminology terms counted: {term_acc.get('total_terms', 0)}")
        print(f"Macro-avg consistency: {_fmt_metric(term_cons.get('macro_avg_consistency'))}")
        print(f"Weighted-avg consistency: {_fmt_metric(term_cons.get('weighted_avg_consistency'))}")

        if term_acc.get('per_term_ratios'):
            print('Top terminology accuracy terms:')
            for term, ratio in _preview_term_stats({
                term: {'occ': 1, 'ratio': ratio}
                for term, ratio in term_acc['per_term_ratios'].items()
            }, limit=TOP_TERM_PREVIEW):
                print(f"  - {term}: ratio={_fmt_metric(ratio['ratio'])}")
        else:
            print('Top terminology accuracy terms: N/A')

        per_term_consistency = term_cons.get('per_term') or {}
        if per_term_consistency:
            print('Top terminology consistency terms:')
            for term, stats in _preview_term_stats(per_term_consistency, limit=TOP_TERM_PREVIEW):
                print(
                    f"  - {term}: occ={stats.get('occ', 0)}, "
                    f"pseudo_ref={stats.get('pseudo_ref')}, "
                    f"consistency={_fmt_metric(stats.get('consistency'))}"
                )
        else:
            print('Top terminology consistency terms: N/A')

        print('Sample previews:')
        for idx, (sample, pred) in enumerate(list(zip(samples, preds))[:TOP_SAMPLE_PREVIEW], start=1):
            ref = sample.get(ref_field)
            print(f'  [{idx}] EN: {sample.get("en", "").strip()}')
            print(f'      PRED: {pred.strip()}')
            print(f'      REF : {ref.strip() if isinstance(ref, str) else ref}')

print('\nEvaluation finished.')

Found data files: ['data/ende_dev.jsonl', 'data/enes_dev.jsonl', 'data/enru_dev.jsonl']

=== File: data/ende_dev.jsonl | target=German | samples=500 | MAX_SAMPLES=None ===

--- Mode: no_term ---


ende_dev.jsonl - no_term:   0%|          | 0/500 [00:00<?, ?it/s]

ende_dev.jsonl - no_term: 100%|██████████| 500/500 [14:39<00:00,  1.76s/it]


BLEU: 23.67
chrF2++: 55.81
Terminology accuracy (ratio %): N/A
Terminology terms counted: 0
Macro-avg consistency: N/A
Weighted-avg consistency: N/A
Top terminology accuracy terms: N/A
Top terminology consistency terms: N/A
Sample previews:
  [1] EN: This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database features.
      PRED: <deu> Diese Dienstleistung beschreibt die bereitgestellte (laufende) Zustand des SAP HANA-Datenbankobjekte, zum Beispiel Tabellen, Ansichten oder Prozeduren, die durch die SAP-Integrated Development Environment (WebIDE)-Editoren als Familie von konsistenten Designzeitobjekten für alle wichtigsten SAP HANA-Plattform-Datenbankfunktionen erstellt oder angepasst wurden.</deu>
      REF : Dieser Service beschreibt 

ende_dev.jsonl - proper_term: 100%|██████████| 500/500 [14:59<00:00,  1.80s/it]


...

ende_dev.jsonl - random_term: 100%|██████████| 500/500 [15:08<00:00,  1.82s/it]


BLEU: 28.81
chrF2++: 59.37
Terminology accuracy (ratio %): 64.65
Terminology terms counted: 627
Macro-avg consistency: 0.95
Weighted-avg consistency: 0.87
Top terminology accuracy terms:
  - 're: ratio=1.00
  - 's: ratio=1.00
  - 01: ratio=1.00
  - 2: ratio=1.00
  - AROs: ratio=1.00
Top terminology consistency terms:
  - following: occ=17, pseudo_ref=folgende, consistency=0.29
  - Select: occ=15, pseudo_ref=Wählen, consistency=0.93
  - work: occ=9, pseudo_ref=arbeiten, consistency=0.44
  - Choose: occ=8, pseudo_ref=Wählen, consistency=1.00
  - data: occ=7, pseudo_ref=Daten, consistency=0.86
Sample previews:
  [1] EN: This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database features.
      PRED: <deu> Diese Dienstleistung beschreibt d

enes_dev.jsonl - no_term: 100%|██████████| 500/500 [13:42<00:00,  1.65s/it]


BLEU: 38.16
chrF2++: 64.16
Terminology accuracy (ratio %): N/A
Terminology terms counted: 0
Macro-avg consistency: N/A
Weighted-avg consistency: N/A
Top terminology accuracy terms: N/A
Top terminology consistency terms: N/A
Sample previews:
  [1] EN: In such cases you may use the Move Items or Merge feature.
      PRED: <es> En tales casos puede usar la función Mover Artículos o Fusionar. </es>
      REF : En estos casos, puede utilizar la función Mover elementos o Fusionar .
  [2] EN: Save and Publish
      PRED: <es> Guardar y Publicar </es>
      REF : Guardar y publicar

--- Mode: proper_term ---


enes_dev.jsonl - proper_term:  82%|████████▏ | 412/500 [11:50<02:01,  1.39s/it]